# Traditional Methods

---
This notebook implements the Hull-White model from the ground up:
1. Yield curve construction
2. Simulation of forward rates using the Hull-White model 
3. Pricing the IRS using closed-form prices
4. Pricing of IRS using Simulated forward rates from the Hull-White 
5. Computation of Expected Exposure of IRS 
6. Computation of CVA and DVA 

In [2]:
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.special import roots_hermitenorm
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import pandas as pd

_ = plt.ioff()

---
## 1. Yield Curve Construction

Zero rates $y(T_j)$ at seven tenor grid points $T_j \in \{1/12, 3/12, 6/12, 9/12, 1, 2, 3, 5, 7, 10\}$ years
are sourced from the Federal Reserve's fitted yield curve dataset (FEDS200628),
which provides daily Svensson model parameters and zero-coupon yields
for US Treasuries from 1961 to the present.

From the market inputs we derive, in order:
- $y(0.5)$: computed from Svensson parameters (not directly in the dataset)
- Discount factors: $P(0, T_j) = \exp(-y(T_j) \cdot T_j)$
- Cubic spline over zero rates for interpolation at arbitrary maturities
- Instantaneous forward rate: $f(0, T) = y(T) + T \cdot y'(T)$
- Initial short rate: $r_0 = \beta_0 + \beta_1$ (limiting value of Svensson curve as $T \to 0$)
- Hull-White components $B(0,T_j)$, $A(0,T_j)$, and drift $\theta(T_j)$ at each tenor

In [3]:
DATA_PATH = 'feds200628.csv'
CURVE_DATE = '2026-03-27'

# ── Load FEDS dataset ─────────────────────────────────────────────────────────
feds = pd.read_csv(DATA_PATH, skiprows=9, na_values='NA')
feds['Date'] = pd.to_datetime(feds['Date'])
feds = feds.set_index('Date')

row = feds.loc[CURVE_DATE]
print(f"Curve date: {CURVE_DATE}")

beta0, beta1, beta2, beta3 = row['BETA0'], row['BETA1'], row['BETA2'], row['BETA3']
tau1,  tau2                = row['TAU1'],  row['TAU2']
print(f"Svensson: β0={beta0:.6f}  β1={beta1:.6f}  β2={beta2:.4f}  β3={beta3:.4f}")
print(f"          τ1={tau1:.4f}   τ2={tau2:.4f}")

def svensson(T, b0, b1, b2, b3, t1, t2):
    """
    Svensson (1994) zero rate (%) at maturity T (years).
    Limit as T->0 is b0+b1, handled explicitly to avoid 0/0.
    """
    T = np.atleast_1d(np.asarray(T, dtype=float))
    safe = np.where(T < 1e-10, 1e-10, T)
    e1   = np.exp(-safe / t1)
    e2   = np.exp(-safe / t2)
    term1 = (1.0 - e1) / (safe / t1)
    term2 = term1 - e1
    term3 = (1.0 - e2) / (safe / t2) - e2
    y     = b0 + b1 * term1 + b2 * term2 + b3 * term3
    return np.where(T < 1e-10, b0 + b1, y)

# ── Tenor grid and zero rates ─────────────────────────────────────────────────
tenors = np.array([1/12, 0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0])
tenor_labels = ['1M', '3M', '6M', '9M', '1Y', '2Y', '3Y', '5Y', '7Y', '10Y']
zero_rates_pct = svensson(tenors, beta0, beta1, beta2, beta3, tau1, tau2)
zero_rates = zero_rates_pct / 100.0
discount_factors = np.exp(-zero_rates * tenors)

# ── Initial short rate: exact Svensson limit ──────────────────────────────────
r0 = (beta0 + beta1) / 100.0

print(f"\nr_0 = β0+β1 = {(beta0+beta1):.4f}% = {r0:.6f}")
print(f"\n{'Tenor':>8}  {'y(T)%':>8}  {'P(0,T)':>10}")
print("-" * 32)
for label, y, P in zip(tenor_labels, zero_rates_pct, discount_factors):
    print(f"{label:>8}  {y:>8.4f}  {P:>10.6f}")

Curve date: 2026-03-27
Svensson: β0=0.000066  β1=3.837997  β2=-87.3371  β3=98.5695
          τ1=10.2507   τ2=11.1148

r_0 = β0+β1 = 3.8381% = 0.038381

   Tenor     y(T)%      P(0,T)
--------------------------------
      1M    3.8371    0.996808
      3M    3.8358    0.990456
      6M    3.8356    0.981005
      9M    3.8374    0.971630
      1Y    3.8409    0.962320
      2Y    3.8709    0.925504
      3Y    3.9218    0.889003
      5Y    4.0664    0.816017
      7Y    4.2417    0.743103
     10Y    4.5143    0.636718


In [ ]:
# Plot the Fed Svensson zero curve out to 10 years
plot_maturities = np.linspace(1/12, 10.0, 400)
plot_zero_rates_pct = svensson(plot_maturities, beta0, beta1, beta2, beta3, tau1, tau2)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(plot_maturities, plot_zero_rates_pct, label='Svensson zero curve', linewidth=2)
ax.scatter(tenors, zero_rates_pct, color='black', s=35, zorder=3, label='Selected tenor nodes')
ax.set_xlim(1/12, 10.0)
ax.set_xticks(tenors)
ax.set_xticklabels(tenor_labels, rotation=45)
ax.set_xlabel('Maturity')
ax.set_ylabel('Zero yield (%)')
ax.set_title(f'Fed Svensson Zero Curve on {CURVE_DATE}')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig('fed_svensson_zero_curve_10y.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved: fed_svensson_zero_curve_10y.png')

In [4]:
# ── Svensson curve as primary interpolant ────────────────────────────────────
# Using Svensson directly (rather than cubic spline) gives a smooth,
# analytically differentiable curve and ensures r_0 = β0+β1 is exactly
# consistent with f(0,0), so the no-arbitrage calibration holds to machine
# precision without flat-extrapolation artefacts.

def zero_rate(T):
    """Svensson zero rate (decimal) at any maturity T."""
    return svensson(T, beta0, beta1, beta2, beta3, tau1, tau2) / 100.0

def discount_factor(T):
    """P(0,T) = exp(-y(T)*T)."""
    T = np.atleast_1d(np.asarray(T, dtype=float))
    return np.exp(-zero_rate(T) * T)

def fwd_rate(T, dt=1e-7):
    """
    f(0,T) = y(T) + T * dy/dT.
    Svensson dy/dT computed via central finite difference.
    """
    T = np.atleast_1d(np.asarray(T, dtype=float))
    Tl = np.maximum(T - dt, 1e-8)
    Tr = T + dt
    dydT = (zero_rate(Tr) - zero_rate(Tl)) / (Tr - Tl)
    return zero_rate(T) + T * dydT

# r_0 = lim_{T->0} y(T) = (β0+β1)/100  (exact Svensson limit)
r0 = (beta0 + beta1) / 100.0
print(f"r_0 = β0 + β1 = {r0*100:.4f}%")
print(f"f(0, 1e-6)    = {fwd_rate(1e-6)[0]*100:.4f}%  (numerical limit check)")

r_0 = β0 + β1 = 3.8381%
f(0, 1e-6)    = 3.8381%  (numerical limit check)


In [6]:
# ── Hull-White parameters ─────────────────────────────────────────────────────
lam   = 0.10    # mean reversion speed  (λ in paper)
eta   = 0.015   # short-rate volatility (η in paper)

# ── B(t, T) ───────────────────────────────────────────────────────────────────
def B(t, T):
    """B(t,T) = (1 - exp(-lam*(T-t))) / lam."""
    return (1.0 - np.exp(-lam * (T - t))) / lam

# ── ln A(t, T) and A(t, T) ────────────────────────────────────────────────────
def lnA(t, T):
    Bt        = B(t, T)
    P0T       = discount_factor(T)
    P0t       = discount_factor(t)
    f0t       = fwd_rate(t)
    convexity = (eta**2 / (4.0 * lam)) * Bt**2 * (1.0 - np.exp(-2.0 * lam * t))
    return np.log(P0T / P0t) + Bt * f0t - convexity

def A(t, T):
    return np.exp(lnA(t, T))

# ── Bond price P(t, T; r_t) ───────────────────────────────────────────────────
def bond_price(t, T, r_t):
    """P(t,T) = A(t,T) * exp(-B(t,T) * r_t)."""
    return A(t, T) * np.exp(-B(t, T) * r_t)

# ── theta(t): Hull-White drift ────────────────────────────────────────────────
def theta(t, dt=1e-5):
    """
    theta(t) = df(0,t)/dt + lam*f(0,t) + eta^2/(2*lam)*(1 - exp(-2*lam*t))
    """
    t    = np.atleast_1d(np.asarray(t, dtype=float))
    f    = fwd_rate(t)
    dfdt = np.where(
        t < tenors[0],
        0.0,
        (fwd_rate(t + dt) - fwd_rate(t - dt)) / (2.0 * dt)
    )
    return dfdt + lam * f + (eta**2 / (2.0 * lam)) * (1.0 - np.exp(-2.0 * lam * t))

In [7]:
# ── Summary table at tenor grid points ────────────────────────────────────────
B0  = B(0.0, tenors)
A0  = A(0.0, tenors)
f0  = fwd_rate(tenors)
th0 = theta(tenors)

df_table = pd.DataFrame({
    'T_j (yrs)'  : tenors,
    'y(T_j) %'   : zero_rates * 100,
    'P(0,T_j)'   : discount_factors,
    'f(0,T_j) %' : f0 * 100,
    'theta(T_j)' : th0,
    'B(0,T_j)'   : B0,
    'A(0,T_j)'   : A0,
})

print(f"Curve date : {CURVE_DATE}")
print(f"r_0        : {r0*100:.4f}%  (β0 + β1 from Svensson, overnight limit)\n")
print(df_table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

Curve date : 2026-03-27
r_0        : 3.8381%  (β0 + β1 from Svensson, overnight limit)

 T_j (yrs)  y(T_j) %  P(0,T_j)  f(0,T_j) %  theta(T_j)  B(0,T_j)  A(0,T_j)
  0.500000  3.835637  0.981005    3.837189    0.003622  0.487706  0.999540
  1.000000  3.840900  0.962319    3.858379    0.004789  0.951626  0.998117
  2.000000  3.870900  0.925503    3.953706    0.005684  1.812692  0.992186
  3.000000  3.921800  0.889004    4.100680    0.006243  2.591818  0.981985
  5.000000  4.066400  0.816017    4.477705    0.006834  3.934693  0.949040
  7.000000  4.241700  0.743104    4.879925    0.007927  5.034147  0.901489
 10.000000  4.514300  0.636717    5.393795    0.007865  6.321206  0.811544


In [8]:
# ── Calibration check: A(0,T)*exp(-B(0,T)*r0) must recover P(0,T) exactly ────
P_model = A0 * np.exp(-B0 * r0)
P_market = discount_factors
errors = np.abs(P_model - P_market)

print("Calibration check: |P_model - P_market|")
for T, err in zip(tenors, errors):
    print(f"  T={T:4.1f}yr  error = {err:.2e}")
print(f"\nMax absolute error: {errors.max():.2e}")

Calibration check: |P_model - P_market|
  T= 0.5yr  error = 1.11e-16
  T= 1.0yr  error = 3.41e-07
  T= 2.0yr  error = 8.93e-07
  T= 3.0yr  error = 3.81e-07
  T= 5.0yr  error = 4.81e-07
  T= 7.0yr  error = 1.57e-06
  T=10.0yr  error = 1.23e-06

Max absolute error: 1.57e-06


In [ ]:
# ── Plots ──────────────────────────────────────────────────────────────────────
T_fine = np.linspace(0.1, 10.0, 500)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Zero rate curve
axes[0].plot(T_fine, zero_rate(T_fine) * 100, 'b-', lw=2, label='Spline')
axes[0].scatter(tenors, zero_rates * 100, color='red', zorder=5, label='Market quotes')
axes[0].set_xlabel('Maturity (yrs)')
axes[0].set_ylabel('Zero rate (%)')
axes[0].set_title('Zero Rate Curve')
axes[0].legend()

# Discount curve
axes[1].plot(T_fine, discount_factor(T_fine), 'g-', lw=2, label='Spline')
axes[1].scatter(tenors, discount_factors, color='red', zorder=5, label='Market quotes')
axes[1].set_xlabel('Maturity (yrs)')
axes[1].set_ylabel('P(0, T)')
axes[1].set_title('Discount Curve')
axes[1].legend()

# Instantaneous forward rate
axes[2].plot(T_fine, fwd_rate(T_fine) * 100, 'm-', lw=2, label='f(0, T)')
axes[2].axhline(r0 * 100, color='gray', ls='--', lw=1, label=f'r_0 = {r0*100:.2f}%')
axes[2].set_xlabel('Maturity (yrs)')
axes[2].set_ylabel('Forward rate (%)')
axes[2].set_title('Instantaneous Forward Rate')
axes[2].legend()

fig.tight_layout()
fig.savefig('yield_curve.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved: yield_curve.png')

---
## 2. IRS Specification and ATM Fixed Rate

We price a spot-starting payer IRS with:
- Notional $N = 1$, maturity $T_n = 10$ years, quarterly payments ($\tau = 0.25$)
- Payment dates $T_j = j \cdot \tau$ for $j = 1, \ldots, n = 40$
- Fixed rate $K^{\text{ATM}}$ set so $V(0) = 0$

Monitoring dates coincide with payment dates. At monitoring date $t_k = k\tau$ (a reset date), the remaining swap value is:
$$V(t_k;\,r_{t_k}) = \bigl(1 - P(t_k, T_n)\bigr) - K^{\text{ATM}}\,\tau \sum_{j=k+1}^{n} P(t_k, T_j)$$
where $P(t_k, T_j) = A(t_k, T_j)\exp(-B(t_k, T_j)\,r_{t_k})$.

In [9]:
# ── IRS contract specification ────────────────────────────────────────────────
Tn      = 10.0                              # swap maturity (years)
tau_pay = 0.25                              # quarterly day count fraction
pay_dates = np.arange(tau_pay, Tn + 1e-10, tau_pay)   # T_1, ..., T_n
n_pay   = len(pay_dates)                    # 40 payment dates
mon_dates = pay_dates                       # monitoring dates = payment dates

# ── ATM fixed rate K^ATM at t=0 ──────────────────────────────────────────────
# K^ATM = (P(0,T_0) - P(0,T_n)) / (tau * sum P(0,T_j))
# T_0 = 0 -> P(0,0) = 1
P0_pay  = discount_factor(pay_dates)        # P(0, T_j) for j=1,...,n
K_atm   = (1.0 - P0_pay[-1]) / (tau_pay * P0_pay.sum())

print(f"Swap maturity : {Tn} years,  quarterly payments,  n = {n_pay}")
print(f"K_ATM         : {K_atm*100:.4f}%")
print(f"Check V(0)    : {1.0 - P0_pay[-1] - K_atm * tau_pay * P0_pay.sum():.2e}  (should be 0)")

Swap maturity : 10.0 years,  quarterly payments,  n = 40
K_ATM         : 4.4744%
Check V(0)    : 0.00e+00  (should be 0)


---
## 3. Model 1a — Closed-Form IRS Valuation

Given the short rate $r_t$ directly, the IRS value is evaluated analytically.
No simulation required: every bond price is a closed-form function of $r_t$.

In [ ]:
def irs_1a(t_k, r_t, k_idx):
    """
    Model 1a: closed-form IRS MtM at monitoring date t_k = pay_dates[k_idx-1].
    Remaining payment dates are pay_dates[k_idx:].

    V(t_k; r_t) = (1 - P(t_k, T_n)) - K_atm * tau * sum_{j > k} P(t_k, T_j)
    """
    remaining = pay_dates[k_idx:]           # payment dates strictly after t_k
    if len(remaining) == 0:
        return 0.0
    P_Tn  = bond_price(t_k, Tn, r_t)
    P_ann = tau_pay * np.sum(bond_price(t_k, remaining, r_t))
    return (1.0 - P_Tn) - K_atm * P_ann

# ── Evaluate V(t; r_t) as a function of r_t at several monitoring dates ───────
r_grid = np.linspace(-0.02, 0.12, 300)
t_plot = [1.0, 3.0, 5.0, 7.0, 9.0]        # monitoring dates to display

fig, ax = plt.subplots(figsize=(10, 5))
for t in t_plot:
    k_idx = int(round(t / tau_pay))        # index into pay_dates
    V = np.array([irs_1a(t, r, k_idx) for r in r_grid])
    ax.plot(r_grid * 100, V, label=f't = {t:.0f}yr')

ax.axhline(0, color='black', lw=0.8, ls='--')
ax.axvline(r0 * 100, color='gray', lw=0.8, ls=':', label=f'r_0 = {r0*100:.2f}%')
ax.set_xlabel('Short rate $r_t$ (%)')
ax.set_ylabel('IRS MtM  $V(t;\,r_t)$')
ax.set_title('Model 1a — Closed-Form IRS Value vs Short Rate')
ax.legend()
fig.tight_layout()
fig.savefig('model_1a_irs.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved: model_1a_irs.png')

# ── Spot check at r_t = r_0 ───────────────────────────────────────────────────
print('V(t, r_0) at each monitoring date (should be near zero at early dates):')
for t in t_plot:
    k = int(round(t / tau_pay))
    print(f'  t={t:.0f}yr  V = {irs_1a(t, r0, k):+.6f}')

---
## 4. Model 1b — Monte Carlo IRS Valuation

The short rate is simulated forward from $r_0$ using the exact Gaussian transition.
The IRS value along each path uses the same closed-form bond price formula as Model 1a,
evaluated at the simulated $r^{(m)}_{t_k}$.

In [ ]:
M_paths = 10_000                            # number of simulation paths
K_mon   = len(mon_dates)                    # 40 monitoring dates

# ── Exact Gaussian short rate simulation ─────────────────────────────────────
np.random.seed(42)
r_paths = np.zeros((M_paths, K_mon))       # shape: (M, K)
r_prev  = np.full(M_paths, r0)            # initialise all paths at r_0

for k, t_next in enumerate(mon_dates):
    t_prev = mon_dates[k - 1] if k > 0 else 0.0
    dt     = t_next - t_prev

    # exact transition parameters
    e_lam  = np.exp(-lam * dt)
    mu     = fwd_rate(t_next)[0] + (r_prev - fwd_rate(t_prev)[0]) * e_lam
    nu     = eta * np.sqrt((1.0 - np.exp(-2.0 * lam * dt)) / (2.0 * lam))

    r_paths[:, k] = mu + nu * np.random.standard_normal(M_paths)
    r_prev = r_paths[:, k]

print(f"Simulated {M_paths:,} paths over {K_mon} monitoring dates")
print(f"r_paths shape: {r_paths.shape}")
print(f"\nShort rate statistics at selected dates:")
print(f"{'Date':>6}  {'Mean%':>8}  {'Std%':>7}  {'f(0,t)%':>8}")
print("-" * 36)
for k in [3, 7, 19, 31, 39]:
    t = mon_dates[k]
    print(f"{t:>6.2f}  {r_paths[:,k].mean()*100:>8.4f}  "
          f"{r_paths[:,k].std()*100:>7.4f}  {fwd_rate(t)[0]*100:>8.4f}")

In [ ]:
# ── IRS MtM along each simulated path (Model 1b) ─────────────────────────────
V_paths = np.zeros((M_paths, K_mon))       # V^(m)(t_k)

for k in range(K_mon):
    t_k       = mon_dates[k]
    remaining = pay_dates[k + 1:]          # payment dates after t_k
    if len(remaining) == 0:
        continue
    r_k   = r_paths[:, k]                  # shape (M,)
    P_Tn  = bond_price(t_k, Tn,       r_k)           # shape (M,)
    P_ann = tau_pay * np.sum(
        [bond_price(t_k, Tj, r_k) for Tj in remaining], axis=0)   # shape (M,)
    V_paths[:, k] = (1.0 - P_Tn) - K_atm * P_ann

print(f"V_paths shape: {V_paths.shape}")
print(f"\nIRS MtM statistics at selected dates:")
print(f"{'Date':>6}  {'Mean':>9}  {'Std':>9}  {'Min':>9}  {'Max':>9}")
print("-" * 50)
for k in [3, 7, 19, 31, 39]:
    v = V_paths[:, k]
    print(f"{mon_dates[k]:>6.2f}  {v.mean():>9.5f}  {v.std():>9.5f}  "
          f"{v.min():>9.5f}  {v.max():>9.5f}")

In [ ]:
# ── Plots: short rate paths + IRS MtM distribution ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Short rate paths (sample of 100)
ax = axes[0]
for m in range(100):
    ax.plot(mon_dates, r_paths[m] * 100, alpha=0.15, lw=0.6, color='steelblue')
ax.plot(mon_dates, r_paths.mean(axis=0) * 100, 'r-', lw=2, label='Path mean')
ax.plot(mon_dates, fwd_rate(mon_dates) * 100, 'k--', lw=1.5, label='f(0,t)')
ax.set_xlabel('Time (years)'); ax.set_ylabel('Short rate (%)')
ax.set_title('Model 1b — Simulated Short Rate Paths')
ax.legend()

# IRS MtM paths (sample of 100)
ax = axes[1]
for m in range(100):
    ax.plot(mon_dates, V_paths[m], alpha=0.15, lw=0.6, color='darkorange')
ax.plot(mon_dates, V_paths.mean(axis=0), 'r-', lw=2, label='Path mean')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('Time (years)'); ax.set_ylabel('IRS MtM  $V^{(m)}(t_k)$')
ax.set_title('Model 1b — IRS MtM along Simulated Paths')
ax.legend()

# Model 1a vs 1b comparison: E[V(t)] across monitoring dates
ax = axes[2]
# Model 1a: V(t, E[r_t]) — closed-form at the mean short rate
V_1a_mean = np.array([irs_1a(mon_dates[k], r_paths[:,k].mean(), k+1)
                       for k in range(K_mon)])
# Model 1b: sample mean of path-wise V
V_1b_mean = V_paths.mean(axis=0)

ax.plot(mon_dates, V_1a_mean, 'b-',  lw=2, label='Model 1a  V(t, $\\bar{r}_t$)')
ax.plot(mon_dates, V_1b_mean, 'r--', lw=2, label='Model 1b  $E[V^{(m)}(t_k)]$')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('Time (years)'); ax.set_ylabel('Mean IRS MtM')
ax.set_title('Model 1a vs 1b — Mean IRS Value')
ax.legend()

fig.tight_layout()
fig.savefig('model_1a_1b_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved: model_1a_1b_comparison.png')